# Silver data cleaning: open charge FI

In [1]:
import pandas as pd
from pathlib import Path

print(f"Current directory: {Path.cwd()}")

Current directory: c:\DE_Course\de-week8-mini-project-team1\Silver


In [6]:
# Read the Bronze-level data, the CSV file into a DataFrame
bronze_df = pd.read_csv("../Bronze/open_charge_raw_FI.csv", encoding="utf-8")

# Inspect the DataFrame
print(bronze_df.shape, "\n")
print(bronze_df.columns, "\n")
print(bronze_df.dtypes)

(4966, 86) 

Index(['UserComments', 'PercentageSimilarity', 'MediaItems',
       'IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'ParentChargePointID', 'DataProviderID', 'DataProvidersReference',
       'OperatorID', 'OperatorsReference', 'UsageTypeID', 'UsageCost',
       'Connections', 'NumberOfPoints', 'GeneralComments', 'DatePlanned',
       'DateLastConfirmed', 'StatusTypeID', 'DateLastStatusUpdate',
       'MetadataValues', 'DataQualityLevel', 'DateCreated',
       'SubmissionStatusTypeID', 'DataProvider.WebsiteURL',
       'DataProvider.Comments',
       'DataProvider.DataProviderStatusType.IsProviderEnabled',
       'DataProvider.DataProviderStatusType.ID',
       'DataProvider.DataProviderStatusType.Title',
       'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
       'DataProvider.IsApprovedImport', 'DataProvider.License',
       'DataProvider.DateLastImported', 'DataProvider.ID',
       'DataProvider.Title', 'OperatorInfo.WebsiteURL',
     

In [10]:
# Define the list of columns to keep and mapping, because we want to rename some columns
column_mapping = {
    "ID": "id",
    "NumberOfPoints": "number_of_points",
    "DateCreated": "date_created", # this one needs to be split later, only one key can exist in a Python dict
    "StatusType.IsOperational": "is_operational",
    "AddressInfo.Town": "municipality",
    "AddressInfo.StateOrProvince": "region",
    "AddressInfo.Country.Title": "country",
    "AddressInfo.Latitude": "latitude",
    "AddressInfo.Longitude": "longitude",
    "fetch_timestamp": "fetch_timestamp"
}

# Select and rename
silver_df = (
    bronze_df[list(column_mapping.keys())]
    .rename(columns=column_mapping)
)

# Convert types
silver_df["date_created"] = pd.to_datetime(silver_df["date_created"])

# Derive columns
silver_df = silver_df.assign(
    year=silver_df["date_created"].dt.year,
    month=silver_df["date_created"].dt.month,
)

# Define the final column order explicitly
silver_df = silver_df[
    [
        "id",
        "number_of_points",
        "year",
        "month",
        "date_created",
        "is_operational",
        "municipality",
        "region",
        "country",
        "latitude",
        "longitude",
        "fetch_timestamp",
    ]
]

# Inspect the silver DataFrame
print(silver_df.shape, "\n")
print(silver_df.columns, "\n")
print(silver_df.dtypes)

(4966, 12) 

Index(['id', 'number_of_points', 'year', 'month', 'date_created',
       'is_operational', 'municipality', 'region', 'country', 'latitude',
       'longitude', 'fetch_timestamp'],
      dtype='str') 

id                                int64
number_of_points                float64
year                              int32
month                             int32
date_created        datetime64[us, UTC]
is_operational                   object
municipality                        str
region                              str
country                             str
latitude                        float64
longitude                       float64
fetch_timestamp                     str
dtype: object


In [ ]:
# Convert the data types to use the nullable Pandas dtypes (Int64, boolean, string) for Silver-layer data
# because they preserve missing values (<NA>)

silver_df["number_of_points"] = silver_df["number_of_points"].astype("Int64")

silver_df["year"] = silver_df["year"].astype("Int64")
silver_df["month"] = silver_df["month"].astype("Int64")

# print(silver_df["is_operational"].unique()) # [True, <NA>, False]
silver_df["is_operational"] = silver_df["is_operational"].astype("boolean")

# use string of Pandas instead of str
string_columns = ["municipality", "region", "country", "fetch_timestamp"]
silver_df[string_columns] = silver_df[string_columns].astype("string")

print(silver_df.dtypes)

id                                int64
number_of_points                  Int64
year                              int32
month                             int32
date_created        datetime64[us, UTC]
is_operational                  boolean
municipality                     string
region                           string
country                          string
latitude                        float64
longitude                       float64
fetch_timestamp                  string
dtype: object
